# 06 Reading Inside the Model: Residual Streams, Per-Layer Logits, and DoLa

If you have already read Tutorial 00 (activation steering) and Tutorial 01 (probes and `@pllm.export`), you might wonder what actually happens at each layer inside the model. In Hugging Face, passing `output_hidden_states=True` gives you every layer's residual stream, which you can multiply by the LM head for a logit lens. In a vLLM serving engine, however, intermediate tensors from concurrent requests are packed together and execution is captured into CUDA graphs, making those intermediate states hard to inspect directly. As a result, you cannot observe layer-by-layer token predictions during serving, nor run decoding algorithms like DoLa that rely on per-layer logits.

In pLLM, doing so takes only two input markers: `pllm.residual(layer)` reads the residual stream, and `pllm.layer_logits(layers)` computes the per-layer logit lens—and both run inside CUDA graphs (with an empty `eager_reason`). Moreover, DoLa is already included as a built-in workload in pLLM in just a few dozen lines of code.

In this example, we walk through three steps:
1. Read the residual stream to answer the question left open in Tutorial 00: "How large should a steering vector be?"
2. Run a logit lens to see at which layer the correct answer emerges.
3. Run DoLa to alter decoding using per-layer logits, and measure both its output changes and its throughput cost.

Let's load the model and tokenizer first.

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # Show only vLLM errors and suppress startup logs

from collections import defaultdict

import torch
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm
from pllm.benchmark import throughput
from pllm.workloads import DoLaWorkload

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=32, max_num_batched_tokens=4096,
                          gpu_memory_utilization=0.5, seed=7)

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)


def chat(question: str) -> str:
    return tokenizer.apply_chat_template([{"role": "user", "content": question}], tokenize=False,
                                         add_generation_prompt=True)


greedy = SamplingParams(max_tokens=32, temperature=0.0)

## Step 1: Residual Stream — How Large Should a Steering Vector Be?

In this step, we answer a practical question left open in Tutorial 00: when doing activation steering, is the steering vector we add actually large or small?

First, let's clarify what the residual stream is in plain terms: after each layer's attention and MLP compute their outputs, they *add* those outputs onto a vector running through the entire model—that vector is the residual stream. The next layer reads from it as input, and when we steer activations, that same stream is what we add our vector into.

In pLLM, `pllm.residual(L)` denotes the residual stream *after* layer `L` (that is, the initial embedding plus the outputs of all sublayers from layer `0` through layer `L`). Note how this differs from `pllm.hidden(L)` used in Tutorial 01: `pllm.hidden(L)` is only the portion newly computed by layer `L`'s MLP before it is added back into the residual stream.

Recall from Tutorial 00 that we added `strength × steering_vector` onto the output of layer 12. That steering vector had norm `7.09`, and we chose strengths of `±2`. The residual stream after layer 12's output corresponds directly to `pllm.residual(12)`. To see how large the added vector is relative to the residual stream itself, let's write a workload that only exports norms: in a single method, we read the residual streams after layers 6, 12, 18, and 23, compute the vector norm at every token position, and export them.

In [3]:
norms: dict[int, list[list[float]]] = defaultdict(list)
WATCHED = (6, 12, 18, 23)


class StreamSize(pllm.Workload):
    @pllm.export(name="stream_norm", max_rows=4, max_cols=4)
    def size(self, r6: torch.Tensor = pllm.residual(6), r12: torch.Tensor = pllm.residual(12),
             r18: torch.Tensor = pllm.residual(18), r23: torch.Tensor = pllm.residual(23)) -> torch.Tensor:
        return torch.stack([r.float().norm(dim=-1) for r in (r6, r12, r18, r23)], dim=-1)

    def on_export(self, event: pllm.io.TraceEvent) -> None:
        for identity, row in zip(event.identities, event.tensor().tolist()):
            norms[identity.prompt_index].append(row)

In [4]:
prompts = [chat("Write one sentence about my new job."), chat("Describe your weekend in one sentence.")]
with pllm.make_llm(CONFIG, workload=StreamSize()) as llm:
    llm.generate(prompts, greedy, use_tqdm=False)
    print("eager 原因:", repr(llm.diagnostics().eager_reason))

rows = torch.tensor([row for index in sorted(norms) for row in norms[index]])
for layer, mean in zip(WATCHED, rows.mean(0).tolist()):
    print(f"第 {layer:2d} 层之后的残差流：平均范数 {mean:5.1f}")
print(f"教程 00 在第 12 层加的向量：范数 7.09 × 强度 2 = {7.09 * 2:.1f}")

eager 原因: ''


第  6 层之后的残差流：平均范数  13.3
第 12 层之后的残差流：平均范数  15.9
第 18 层之后的残差流：平均范数  35.5
第 23 层之后的残差流：平均范数  55.2
教程 00 在第 12 层加的向量：范数 7.09 × 强度 2 = 14.2


From the output, the residual stream norm grows steadily in deeper layers: the mean norm is `13.3` after layer 6, `15.9` after layer 12, `35.5` after layer 18, and `55.2` by layer 23. In Tutorial 00, adding our steering vector at layer 12 with strength `2` added a vector of norm `7.09 × 2 = 14.2`—almost as large as the mean norm of the layer-12 residual stream itself (`15.9`). That gives a concrete reference point: strength `2` roughly overhauls the layer-12 residual stream (and as Tutorial 00 noted, pushing a small model too hard causes repetition). Adding a vector of the same magnitude at a deeper layer would be much smaller relative to the residual stream there. Also note that `eager_reason` is `''` (printed as `eager 原因: ''` above), confirming that reading the residual stream does not require disabling CUDA graphs.

In [5]:
LAYERS = tuple(range(24))
lens: dict[int, list[list[int]]] = defaultdict(list)


class LogitLens(pllm.Workload):
    @pllm.export(name="lens", max_rows=4, max_cols=len(LAYERS))
    def top1(self, logits: torch.Tensor = pllm.layer_logits(LAYERS)) -> torch.Tensor:
        return logits.argmax(dim=-1).float()  # Most likely token at each layer, [rows, layers]

    def on_export(self, event: pllm.io.TraceEvent) -> None:
        for identity, row in zip(event.identities, event.tensor().long().tolist()):
            lens[identity.prompt_index].append(row)

In [6]:
with pllm.make_llm(CONFIG, workload=LogitLens()) as llm:
    seven = SamplingParams(max_tokens=7, temperature=0.0)  # Answer is exactly 7 tokens
    output = llm.generate([chat("What is the capital of France? Answer in one sentence.")], seven, use_tqdm=False)[0]
    print("eager 原因:", repr(llm.diagnostics().eager_reason))

generated = list(output.outputs[0].token_ids)
print("回答:", repr(output.outputs[0].text))
# Row 0 comes from the last prompt token; row k comes from the step taking the k-th generated token as input;
# the final layer in row k predicts the k-th generated token (0-indexed).
print("最后一层的预测与实际生成一致:", all(row[-1] == generated[k] for k, row in enumerate(lens[0])))

step = next(k for k, row in enumerate(lens[0]) if tokenizer.decode(row[-1]) == " Paris")
print(f"预测 ' Paris' 的那一步，每一层最可能的 token：")
for layer, token in zip(LAYERS, lens[0][step]):
    print(f"  第 {layer:2d} 层: {tokenizer.decode(token)!r}")

eager 原因: ''


回答: 'The capital of France is Paris.'
最后一层的预测与实际生成一致: True
预测 ' Paris' 的那一步，每一层最可能的 token：
  第  0 层: '\';";\n'
  第  1 层: '\';";\n'
  第  2 层: ".','"
  第  3 层: ".','"
  第  4 层: ".','"
  第  5 层: '\';";\n'
  第  6 层: "',...\n"
  第  7 层: "',...\n"
  第  8 层: '","#'
  第  9 层: 'announcement'
  第 10 层: 'announcement'
  第 11 层: "',...\n"
  第 12 层: 'уницип'
  第 13 层: '那'
  第 14 层: '\';";\n'
  第 15 层: ';break'
  第 16 层: 'MAND'
  第 17 层: 'MAND'
  第 18 层: ';break'
  第 19 层: '家都知道'
  第 20 层: ' capitals'
  第 21 层: ' Paris'
  第 22 层: ' Paris'
  第 23 层: ' Paris'


The model's generated answer is `'The capital of France is Paris.'`. In the output, `True` for matching the final layer's predictions against the actual generated tokens confirms the alignment described above: the final layer's prediction in each row indeed corresponds to the next token actually generated.

Looking closely at the step that predicts `' Paris'`: from layer 0 through layer 19, the top-1 token at each layer consists of fragments unrelated to the question, such as `'\';";\n'`, `'announcement'`, and `';break'`; at layer 20 it shifts to the topically relevant `' capitals'`; and starting at layer 21, the top-1 token becomes `' Paris'` and stays there through layer 23 (the final layer). In this example, the answer takes shape only in the last few layers. Once again, `eager_reason` is empty.

## Step 3: DoLa — Altering Decoding with Per-Layer Logits

Since shallow layers only carry surface fragments while factual answers emerge in the final layers, can we exploit this contrast between shallow and deep layers to improve decoding? That is what we test next with DoLa (Chuang et al., 2024, *"Decoding by Contrasting Layers"*).

In plain terms, DoLa's idea is straightforward: at each generated token, pick from a set of candidate shallow layers the layer whose prediction diverges most from the final layer (measured by Jensen–Shannon divergence, where larger divergence means greater disagreement). Then re-score the vocabulary by subtracting that shallow layer's log-probabilities from the final layer's log-probabilities. To avoid picking tokens that were implausible to begin with, selection is restricted to tokens that the final layer considers sufficiently probable (probability at least `0.1×` the final layer's maximum probability), masking out all others. The intuition is to suppress surface patterns already guessable in shallow layers while boosting information newly formed in deep layers.

pLLM ships this algorithm as `pllm.workloads.DoLaWorkload` (implemented in `pllm/workloads/dola.py`, with about a dozen lines in its core method): it uses a `@pllm.add_logits` method reading `pllm.layer_logits`, with default candidate shallow layers `0, 2, ..., 14`. To use it, simply pass `DoLaWorkload()` to `make_llm`.

Below we first compare greedy answers from native decoding and DoLa on 4 factual questions, then measure the throughput cost. Here we focus only on output differences and runtime cost, without evaluating whether DoLa improves model accuracy.

Before launching the engine, let's inspect DoLa's plan and check its correctness on CPU. The `forward` line in `explain` tells us whether the model stays in CUDA graphs, `flops_per_token_estimate` combines DoLa's extra per-token compute with the base model, and `testing.check` verifies that DoLa's row-wise math does not mix rows across concurrent requests. DoLa has no strength toggle, so we omit `neutral=` here.

In [7]:
from pllm import testing

report = pllm.explain(DoLaWorkload(), CONFIG)
print("CUDA graph:", "保留" if report.eager_reason == "" else report.eager_reason)
print(f"每 token FLOP 估计：{report.flops_per_token_estimate:.3g}（底座模型约 9.88e+08）")
print(testing.check(DoLaWorkload(), CONFIG))

CUDA graph: 保留
每 token FLOP 估计：3.44e+09（底座模型约 9.88e+08）
status  contract / reason
PASS    tensor_parallel_size=1: admission
PASS    compile
PASS    row permutation
PASS    request isolation
PASS    mixed prefill/decode
PASS    chunking / tokens=
PASS    preempted
SKIP    publish_lag: no row method reads versioned state
SKIP    neutral: pass neutral= with the author's disabled settings


In [8]:
questions = ["What is the capital of Australia?", "Who wrote the novel 1984?", "How many legs does a spider have?",
             "What is the boiling point of water at sea level in Fahrenheit?"]
answer = SamplingParams(max_tokens=40, temperature=0.0)


def answers(llm) -> list[str]:
    return [output.outputs[0].text for output in llm.generate([chat(q) for q in questions], answer, use_tqdm=False)]


with pllm.make_llm(CONFIG) as plain:
    plain_texts = answers(plain)
with pllm.make_llm(CONFIG, workload=DoLaWorkload()) as llm:
    dola_texts = answers(llm)

for question, a, b in zip(questions, plain_texts, dola_texts):
    print(question)
    print("  原生:", repr(a))
    print("  DoLa:", repr(b))


What is the capital of Australia?
  原生: 'The capital of Australia is Canberra.'
  DoLa: 'The capital of Australia is Canberra. Canberra is the capital city of Australia.'
Who wrote the novel 1984?
  原生: 'The novel "1984" was written by George Orwell.'
  DoLa: '1984 is a work of fiction, created by George Orwell. The novel is part of Orwell\'s complete series of books titled "1984" and is set in a dystopian'
How many legs does a spider have?
  原生: 'A spider has eight legs.'
  DoLa: 'A spider has eight legs.'
What is the boiling point of water at sea level in Fahrenheit?
  原生: 'The boiling point of water at sea level in Fahrenheit is 212°F.'
  DoLa: 'The boiling point of water at sea level is 212 degrees Fahrenheit.'


Now let's measure the throughput cost. The 0.5B model is sufficient for the inspections above, whereas for throughput we switch to `Qwen/Qwen2.5-7B-Instruct`: a single decode step on 0.5B takes only a few milliseconds, exaggerating the relative share of the LM head and sampler, while 7B is a more typical deployment size (~15 GiB in bf16 weights, requiring a 24 GiB GPU). We test 32 concurrent requests (8 prompts × 4 samples each × 256 generated tokens) using `pllm.benchmark.throughput`: 1 warmup round followed by 3 timed rounds taking the median, where each round times the full `generate` call (including prefill). `DoLaWorkload`'s default candidate shallow layers `0, 2, ..., 14` also apply to the 7B model (28 layers).

In [9]:
BIG = pllm.ModelConfig("Qwen/Qwen2.5-7B-Instruct", max_model_len=512, max_num_seqs=32,
                       max_num_batched_tokens=4096, gpu_memory_utilization=0.85, seed=7)
many = [f"Explain the mathematical idea numbered {i}:" for i in range(8)]
sampling = SamplingParams(n=4, max_tokens=256, temperature=0.8, top_p=0.95, ignore_eos=True, seed=7)

with pllm.make_llm(BIG) as plain:
    plain_rate = throughput(plain, many, sampling)
with pllm.make_llm(BIG, workload=DoLaWorkload()) as llm:
    dola_rate = throughput(llm, many, sampling)
    eager_reason = llm.diagnostics().eager_reason
print(f"7B 吞吐（32 个并发请求，各 256 token）：原生 {plain_rate:.0f} token/s，DoLa {dola_rate:.0f} token/s，"
      f"为原生的 {dola_rate / plain_rate:.0%}；eager 原因 {eager_reason!r}")


7B 吞吐（32 个并发请求，各 256 token）：原生 1727 token/s，DoLa 1140 token/s，为原生的 66%；eager 原因 ''


First, looking at the output changes: out of the 4 questions, DoLa changed the answer on 3 of them. On the capital of Australia, DoLa appended a redundant sentence after answering Canberra; on the author of *1984*, DoLa's answer grew longer and introduced an inaccurate claim (saying the book belongs to a series by Orwell called "1984"); on how many legs a spider has, both answers were identical; and on the boiling point of water, DoLa rephrased `212°F` as `212 degrees Fahrenheit`. Of course, 4 examples cannot tell us whether DoLa makes this 0.5B model more accurate or worse overall.

Next, looking at the throughput cost (7B, 32 concurrent requests, 256 tokens each; RTX 4090, vLLM 0.32.0, bf16, CUDA graphs, median of 3 timed rounds after 1 warmup): in this run, native throughput is `1727 token/s`, and DoLa reaches `1140 token/s`, or `66%` of native. `eager_reason` remains `''`, meaning DoLa runs inside CUDA graphs throughout. The throughput drop comes from genuine compute: for every generated token, FP32 logits and divergences over the 152,064-token vocabulary must be computed across 9 layers (8 candidate shallow layers plus the final layer). The `explain` estimate reflects this directly: on 7B, DoLa increases per-token compute from ~14.1B FLOPs to ~24.0B FLOPs (`1.7×` the base model). On 0.5B, the same DoLa workload is `3.5×` the base model (cell 11 output: ~0.99B to ~3.44B FLOPs), because the vocabulary projection size does not shrink with model depth—the smaller the model, the larger fraction of a step it occupies, so DoLa's relative overhead looks much heavier on small models.

## What the Framework Does Under the Hood

Why can reading residual streams and computing per-layer logits both stay inside CUDA graphs without perturbing the model's own outputs?

1. **Residual stream reads**: The framework attaches reads onto the norm operations already invoked at every layer, copying out the requested rows directly inside the CUDA graph.
2. **Per-layer logits computation**: At the LM head, the framework runs a separate matrix multiplication to compute logits for the selected layers, completely decoupled from the model's own final-logits path—so the model's own logits remain bit-for-bit identical to native vLLM.

## Summary

- `pllm.residual(layer)` reads the residual stream after a specified layer; comparing norms directly shows how large an activation-steering vector is relative to the residual stream itself.
- `pllm.layer_logits(layers)` provides intermediate-layer logits without breaking CUDA graphs, enabling both logit-lens inspection of where answers form and decoding algorithms that depend on per-layer logits.
- `pllm.workloads.DoLaWorkload` implements DoLa's shallow-vs-deep logit contrast decoding in about a dozen lines of core code; it executes inside CUDA graphs, achieving `66%` of native throughput on 7B due to computing FP32 full-vocabulary logits and divergences across multiple layers on every token.

In the next example, [07 Custom Residual Connections](07_custom_residual.ipynb), we move from merely reading the residual stream to rewriting how residual updates themselves are computed (`@pllm.custom_residual`).